# Phase 2 (baselines): phase2_ptbxl_pretraining

One run does all of it for each chosen input variant:

1. preprocess PTB-XL (once per session),
2. pretrain the model on PTB-XL beats (3 seeds),
3. fine-tune on MIT-BIH DS1 from the pretrained weights (6 runs),
4. train the same model from scratch for comparison (6 runs).

The letters are the same four input variants as in phase1_input_representation (A, B, C, D).

Before running (details in `kaggle/README.md`):

1. Settings: **Internet on**. Accelerator: **GPU**.
2. Add-ons > Secrets: `GITHUB_TOKEN` (Contents: read and write on the repo), ticked for this notebook.
3. Add Input: the team's `mitdb-raw` dataset (the notebook finds it under `/kaggle/input`; MIT-BIH is never
   downloaded, and the run stops if the files are missing or differ from the committed checksums).
   Also add a PTB-XL dataset (with `ptbxl_database.csv` and `records500/`); it is found under
   `/kaggle/input`. Without one, set `PTBXL_SOURCE = "download"` to fetch it from PhysioNet (1.8 GB).
4. Set the parameters in the next cell (including `GIT_NAME` and `GIT_EMAIL`), then Run All.
5. When it finishes, the results are already on GitHub: in your clone, `git pull`.
   (Backup: the zip in Output, brought in with `python kaggle/ingest_runs.py <zip>`.)

In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
VARIANTS = ["C"]                      # the variant the phase 1 gate chose (see python -m ecgaug.workflow status)
PREVIEW = False                       # True = run before the phase 1 gate or for another variant; marked provisional
PTBXL_SOURCE = "/kaggle/input"        # searched for the attached PTB-XL dataset (ptbxl_database.csv); "download" = fetch it from PhysioNet
SMOKE_TEST = False                    # True = quick check on fake data (about a minute, no results)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # searched for the attached mitdb-raw dataset; MIT-BIH is never downloaded
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"
PUSH_RESULTS = False                  # True = commit and push the results to GIT_REF from Kaggle; default: ingest the zip yourself
GIT_NAME = ""                         # your name for the results commit, e.g. "Akash Saha"
GIT_EMAIL = ""                        # your GitHub email for the results commit

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
# Turn the variant letters into config files, e.g. "A" -> configs/phase2_ptbxl_pretraining/A_short_window.yaml
configs = []
for letter in VARIANTS:
    (path,) = sorted((REPO / "configs/phase2_ptbxl_pretraining").glob(f"{letter}_*.yaml"))
    configs.append(str(path.relative_to(REPO)))
print("configs:", configs)

extra = ["--device", DEVICE] + (["--ptbxl-download"] if PTBXL_SOURCE == "download" else ["--ptbxl-root", PTBXL_SOURCE])
if SMOKE_TEST:
    extra.append("--smoke")
if PREVIEW:
    extra.append("--preview")
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase2_ptbxl_pretraining", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
cmd += ["--raw-dir", RAW_DATASET_DIR]   # the MIT-BIH files are copied from here and checked against the committed checksums
if PUSH_RESULTS and not SMOKE_TEST:
    branch = GIT_REF if len(GIT_REF) != 40 else "kaggle-results"
    cmd += ["--push-to", branch, "--git-name", GIT_NAME, "--git-email", GIT_EMAIL]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode   # env carries the token for the push
print("finished with exit code", code, "(2 = the run was refused, read the message above)")